# Lektion 16: Werkzeuge & Agenten



| | |
|---|---|
| **Zeit** | ~60 Minuten |
| **Schwierigkeit** | Mittel |
| **Voraussetzungen** | Lektion 13–15 |
| **Du baust** | Einen Agenten, der selbst entscheidet, welches Werkzeug er benutzt – und den Loop sichtbar machst |

## 🎯 Nach dieser Lektion kannst du:

- Tools mit `@tool` definieren (der Docstring ist die Bedienungsanleitung!)

- einen Agenten mit `create_agent` bauen (aktuelle API)

- den Agenten-Loop an den Nachrichtentypen ablesen

- Fehler in Tools abfangen (freundliche Rückgabe statt Crash)

- das Tool-Calling offline mit der Simulation üben

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🧠 Konzept: Der Werkzeugkasten

**WARUM:** Ohne Tools kann ein LLM nur *reden*. Mit Tools *handelt* es: rechnen, nachschlagen, Uhrzeit lesen.

**WAS:** Ein **Agent** erhält Python-Funktionen als **Werkzeuge** und entscheidet **selbst**, wann er welches benutzt – in einer Schleife: *planen → Werkzeug wählen → ausführen → beobachten → antworten*.

**Metapher:** Der neue Mitarbeiter bekommt einen Werkzeugkasten. Auf jedem Werkzeug steht (**Docstring!**), wofür es da ist. **Der Docstring ist die Bedienungsanleitung** – danach wählt der Agent!

> **Benötigte Modell-Fähigkeit:** Tool-Calling. Nicht jedes (lokale) Modell beherrscht das – Offline übst du mit der Simulation, die den kompletten Loop zeigt (siehe `docs/local-ai-server.md`).

In [2]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from lernakademie.llm_client import get_chat_model

@tool
def wortzaehler(text: str) -> str:
    """Zaehlt die Woerter eines Texts."""
    return f"Der Text enthaelt {len(text.split())} Woerter."

agent = create_agent(
    model=get_chat_model(),
    tools=[wortzaehler],
    system_prompt="Du bist ein Assistent fuer Text-Statistiken.",
)

frage = ("Wie viele Woerter hat der Satz 'LangGraph macht Workflows steuerbar'? "
         "Nutze den wortzaehler.")
ergebnis = agent.invoke({"messages": [{"role": "user", "content": frage}]})
print("Finale Antwort:", ergebnis["messages"][-1].content[:150])

Finale Antwort: [OFFLINE] Finale Antwort auf Basis der Tool-Ergebnisse: Der Text enthaelt 13 Woerter.


## 🔎 Den Agenten-Loop SEHEN

Der Agent ist kein Black Box: In `ergebnis["messages"]` steht **jeder Schritt**. Lerne die Typen lesen – das ist dein Fenster in die Entscheidungen:

In [3]:
for nachricht in ergebnis["messages"]:
    info = f"type={type(nachricht).__name__}"
    if getattr(nachricht, "tool_calls", None):
        info += f"  tool_calls={[c['name'] for c in nachricht.tool_calls]}"
    if type(nachricht).__name__ == "ToolMessage":
        info += f"  ergebnis={str(nachricht.content)[:40]}"
    print(info)

type=HumanMessage
type=AIMessage  tool_calls=['wortzaehler']
type=ToolMessage  ergebnis=Der Text enthaelt 13 Woerter.
type=AIMessage


### 🔎 Was sehen wir hier?
1. `HumanMessage` – deine Frage
2. `AIMessage` **mit `tool_calls`** – der Agent ENTSCHEIDET sich für das Werkzeug
3. `ToolMessage` – das Werkzeug-Ergebnis (die Beobachtung!)
4. finale `AIMessage` – die Antwort **auf Basis der Beobachtung**

Das ist planen → handeln → beobachten → antworten – **sichtbar gemacht**.

## 🧠 Konzept: Sichere Tools

**WARUM:** Ein Tool, das bei Müll-Eingaben crasht, bringt den ganzen Agenten zum Absturz. Und: **niemals `eval` mit Nutzereingaben!** Wir bauen einen sicheren Mini-Rechner per Regex.

In [4]:
import re

RECHNUNG = re.compile(r"^\s*(-?\d+(?:\.\d+)?)\s*([+\-*/])\s*(-?\d+(?:\.\d+)?)\s*$")

@tool
def taschenrechner(expression: str) -> str:
    """Berechnet einen einfachen Ausdruck der Form 'zahl operator zahl', z.B. '12 * 7'."""
    match = RECHNUNG.match(expression)
    if not match:
        return "Ungueltiger Ausdruck. Format: 'zahl operator zahl', z.B. '12 * 7'."
    links, operator, rechts = float(match.group(1)), match.group(2), float(match.group(3))
    try:
        if operator == "+": ergebnis = links + rechts
        elif operator == "-": ergebnis = links - rechts
        elif operator == "*": ergebnis = links * rechts
        else: ergebnis = links / rechts
    except ZeroDivisionError:
        return "Division durch Null ist nicht moeglich."
    zahl = int(ergebnis) if ergebnis == int(ergebnis) else round(ergebnis, 6)
    return f"Ergebnis von {expression} = {zahl}"

print(taschenrechner.invoke({"expression": "12 * 7"}))
print(taschenrechner.invoke({"expression": "abc"}))       # freundliche Meldung
print(taschenrechner.invoke({"expression": "12 / 0"}))    # abgefangen

Ergebnis von 12 * 7 = 84
Ungueltiger Ausdruck. Format: 'zahl operator zahl', z.B. '12 * 7'.
Division durch Null ist nicht moeglich.


## ✏️ Deine Aufgabe: Dein eigenes Werkzeug

Du schreibst ein robustes Tool und gibst es einem Agenten.

**Anforderungen:**

- Schreibe ein Tool `uhrzeit()` (kein Parameter), das die aktuelle Uhrzeit als `"Es ist HH:MM Uhr."` zurückgibt (nutze `datetime`).

- Schreibe ein Tool `zufallszahl(minimum: int, maximum: int)`, das eine Zufallszahl zurückgibt – mit Absicherung: bei `minimum > maximum` eine klare Fehlermeldung als Text (kein Crash!).

- Baue einen Agenten mit beiden Tools und stelle eine Frage, die das Uhrzeit-Tool auslöst.

**Schreibe deinen Code in die folgende Zelle:**

In [5]:
from datetime import datetime
import random

@tool
def uhrzeit() -> str:
    """Liefert die aktuelle Uhrzeit im Format HH:MM."""
    return f"Es ist {datetime.now().strftime('%H:%M')} Uhr."

@tool
def zufallszahl(minimum: int, maximum: int) -> str:
    """Zieht eine Zufallszahl zwischen minimum und maximum (inklusive)."""
    if minimum > maximum:
        return "Fehler: minimum muss kleiner oder gleich maximum sein."
    return f"Zufallszahl zwischen {minimum} und {maximum}: {random.randint(minimum, maximum)}"

werk_agent = create_agent(
    model=get_chat_model(),
    tools=[uhrzeit, zufallszahl],
    system_prompt="Du bist ein Alltags-Assistent und nutzt deine Werkzeuge.")

lauf = werk_agent.invoke({"messages": [{"role": "user",
                            "content": "Wie spat ist es? Nutze die uhrzeit."}]})
print(lauf["messages"][-1].content[:120])

[OFFLINE] Finale Antwort auf Basis der Tool-Ergebnisse: Es ist 22:51 Uhr.


**💡 Warum diese Loesung funktioniert:**

Die Absicherung **im** Tool ist der entscheidende Punkt: Der Agent kann die Meldung lesen und dem Nutzer erklären – der Loop läuft weiter statt zu crashen. Im Offline-Modus simuliert die Akademie die Tool-Auswahl (per Tool-Namen in der Frage) – der komplette Ablauf bleibt gleich.

### ✅ Richtig, wenn:

- [ ] Beide Tools haben aussagekräftige Docstrings (deutsch oder englisch – klar!).

- [ ] `zufallszahl(9, 3)` liefert eine Fehlermeldung als String – kein Crash.

- [ ] Der Agent-Ablauf enthält eine ToolMessage (Loop sichtbar).

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [6]:
u = uhrzeit.invoke({})
assert "Uhr" in u, f"uhrzeit() soll 'Es ist HH:MM Uhr.' liefern: {u}"
assert "9" in zufallszahl.invoke({"minimum": 1, "maximum": 9}) or True  # Format-Check:
assert "Fehler" in zufallszahl.invoke({"minimum": 9, "maximum": 3}) or "muss" in zufallszahl.invoke({"minimum": 9, "maximum": 3}).lower()
print("Alle Prüfungen bestanden! ✅")

Alle Prüfungen bestanden! ✅


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Uhrzeit formatieren: `datetime.now().strftime("%H:%M")`.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Absicherung VOR dem Würfeln: `if minimum > maximum: return "Fehler: minimum muss kleiner gleich maximum sein."`.
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Beim Agenten: `create_agent(model=get_chat_model(), tools=[uhrzeit, zufallszahl], system_prompt=...)`.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Wer führt die Berechnung aus – das Modell oder dein Python? Was heißt das für die Zuverlässigkeit?

2. Wann wäre ein fester Graph (wie in Lektion 15) besser als ein freier Agent? Nenne je ein Beispiel.

## ➡️ Weiter

**Weiter mit Lektion 17:** `notebooks/09_agents/17_gedaechtnis_und_freigabe.ipynb` – Gedächtnis über Aufrufe hinweg (Checkpointer) und die Freigabe-Mechanik isoliert.